In [10]:
# !pip install datasets
# !pip install transformers[torch]

In [11]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, losses, InputExample
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import Dataset
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_similarity
import gc


In [12]:
def print_gpu_mem(label=""):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    print(f"[{label}] Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB")

# Example:
print_gpu_mem("Before clearing")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print_gpu_mem("After clearing")


[Before clearing] Allocated: 856.32 MB | Reserved: 11842.00 MB
[After clearing] Allocated: 856.32 MB | Reserved: 942.00 MB


In [13]:
import os
os.environ["WANDB_DISABLED"] = "true"


base_model = 'sentence-transformers/all-mpnet-base-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [14]:
df = pd.read_json('synthetic_data_for_contrastive_learning.jsonl', lines=True)
df.head()

,model_name,anchor_story,similar_story,dissimilar_story
0,meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo,"A mysterious individual, known only by their a...","In the secluded hamlet of Ravenshire, a myster...","In the coastal city of Tidal Cove, a reclusive..."
1,gpt-4o,A mysterious drifter arrives in the lawless fr...,A lone wanderer arrives in the turbulent minin...,"In a sprawling, rain-soaked city, a quiet mech..."
2,OpenAI GPT4o Mini,"A team of paranormal investigators, led by sea...","A group of spectral researchers, led by experi...","In a bustling modern city, a group of amateur ..."
3,OpenAI GPT 5 Chat,A prolonged drought devastates a rural farming...,A severe heatwave grips the remote farming set...,"In a remote coastal town, a series of mysterio..."
4,meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo,The film revolves around a Marine who is sever...,The film follows Gunnery Sergeant Ryder Thomps...,"In a dystopian future, Captain Rachel Kim, a r..."


In [15]:
def load_and_prepare_data(data_path='synthetic_data_for_contrastive_learning.jsonl'):
    """
    Load data and convert to training format

    Args:
        data_path: Path to CSV file
        format_type: 'format1' (triplet) or 'format2' (pairwise)

    Returns:
        List of InputExample objects
    """
    df = pd.read_json('synthetic_data_for_contrastive_learning.jsonl', lines=True)
    syn_df = pd.read_json('gemini_synthetic_data.jsonl', lines=True)
    syn_df_2 = pd.read_json('gemini_synthetic_data_p2.jsonl', lines=True)
    syn_df_3 = pd.read_json('gemini_synthetic_data_p3.jsonl', lines=True)
    syn_df = pd.concat([syn_df, syn_df_2,syn_df_3])
    df = df.dropna(subset=['anchor_story', 'similar_story', 'dissimilar_story'])
    syn_df = syn_df.dropna(subset=['anchor_text', 'text_a', 'text_b','text_a_is_closer'])

    print(f"Loaded {len(df)} examples from {data_path}")

    examples = []

    for i, row in df.iterrows():
        example = InputExample(
            texts=[row['anchor_story'], row['similar_story'], row['dissimilar_story']]
        )
        examples.append(example)

    for i, row in syn_df.iterrows():
      if(row['text_a_is_closer'] == True):
        example = InputExample(texts=[row['anchor_text'], row['text_a'], row['text_b']])
      else:
        example = InputExample(texts=[row['anchor_text'], row['text_b'], row['text_a']])
      examples.append(example)

    print(f"Created {len(examples)} training examples")

    return examples

In [16]:
def create_train_val_split(examples, val_size=0.15, random_state=42):
    """Split data into train and validation sets"""
    train_examples, val_examples = train_test_split(
        examples,
        test_size=val_size,
        random_state=random_state
    )
    print(f"Train: {len(train_examples)}, Val: {len(val_examples)}")
    return train_examples, val_examples

In [17]:
def fine_tune(train_examples, output_path, epochs, batch_size, warmup_steps):
    """
    Fine-tune the model using triplet loss
    """
    print("\n" + "="*60)
    print("Starting Fine-Tuning")
    print("="*60)

    # Load base model
    print(f"Loading base model: {base_model}")
    model = SentenceTransformer(base_model, device=device)

    # Create dataloader
    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=batch_size
    )

    # Define loss function (Triplet Loss)
    train_loss = losses.TripletLoss(model,distance_metric=losses.TripletDistanceMetric.COSINE,triplet_margin=-0.4)

    # Calculate training steps
    steps_per_epoch = len(train_dataloader)
    total_steps = steps_per_epoch * epochs

    print(f"\nTraining Configuration:")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Total steps: {total_steps}")
    print(f"  Warmup steps: {warmup_steps}")

    # Fine-tune with lower learning rate
    print("\nTraining...")
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={'lr': 5e-6},  # ADD THIS - much lower than default 2e-5
        output_path=output_path,
        show_progress_bar=True,
        save_best_model=True
    )

    print(f"\nModel saved to: {output_path}")
    return model

In [18]:

def evaluate_on_test(model_path, test_data_path):
    """
    Evaluate fine-tuned model on test set

    Args:
        model_path: Path to fine-tuned model
        val: Path to test CSV (format2)
    """
    from sklearn.metrics.pairwise import cosine_similarity

    print("\n" + "="*60)
    print("Evaluating Fine-tuned Model")
    print("="*60)

    # Load model
    model = SentenceTransformer(model_path, device=device)

    # Load test data
    df = pd.read_json(test_data_path, lines=True)
    print(f"Test set size: {len(df)}")

    correct = 0
    predictions = []

    for idx, row in df.iterrows():
        # Encode
        embeddings = model.encode([
            row['anchor_text'],
            row['text_a'],
            row['text_b']
        ])

        # Calculate similarities
        sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
        sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]

        # Predict
        pred = sim_a > sim_b
        predictions.append(pred)

        if pred == row['text_a_is_closer']:
            correct += 1

    accuracy = correct / len(df)

    print(f"\n✓ Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Correct: {correct}/{len(df)}")

    return accuracy, predictions

In [19]:
"""
Complete pipeline: train, evaluate, and compare with baseline
"""


# 1. Load and prepare training data
print("\n[1/5] Loading training data...")
train_examples = load_and_prepare_data()

# 2. Split into train/val
print("\n[2/5] Splitting data...")
train_examples, val_examples = create_train_val_split(train_examples)

# 3. Evaluate baseline (before fine-tuning)
print("\n[3/5] Evaluating baseline model...")
baseline_model = SentenceTransformer(base_model)

# from sklearn.metrics.pairwise import cosine_similarity
test_data_path = 'dev_track_a.jsonl'
df_test = pd.read_json(test_data_path, lines=True)

correct_baseline = 0
for idx, row in df_test.iterrows():
    embeddings = baseline_model.encode([row['anchor_text'], row['text_a'], row['text_b']])
    sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]
    if (sim_a > sim_b) == row['text_a_is_closer']:
        correct_baseline += 1

baseline_acc = correct_baseline / len(df_test)
print(f"Baseline accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")

# 4. Fine-tune
print("\n[4/5] Fine-tuning model...")
finetuned_model = fine_tune(
    train_examples,
    output_path='./finetuned_narrative_model',
    epochs=1,
    batch_size=8,
    warmup_steps=100
)

# 5. Evaluate fine-tuned model
print("\n[5/5] Evaluating fine-tuned model...")
finetuned_acc, _ = evaluate_on_test(
    './finetuned_narrative_model',
    test_data_path
)

# Summary
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
print(f"Baseline:    {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"Fine-tuned:  {finetuned_acc:.4f} ({finetuned_acc*100:.2f}%)")
improvement = (finetuned_acc - baseline_acc) * 100
print(f"Improvement: {improvement:+.2f} percentage points")

if finetuned_acc > baseline_acc:
    print(f"\n Fine-tuning improved performance!")
else:
    print(f"\n Fine-tuning did not improve performance")
    print("   Consider: more data, different hyperparameters, or data quality issues")



[1/5] Loading training data...
Loaded 1897 examples from synthetic_data_for_contrastive_learning.jsonl
Created 4870 training examples

[2/5] Splitting data...
Train: 4139, Val: 731

[3/5] Evaluating baseline model...
Baseline accuracy: 0.6150 (61.50%)

[4/5] Fine-tuning model...

Starting Fine-Tuning
Loading base model: sentence-transformers/all-mpnet-base-v2


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



Training Configuration:
  Epochs: 1
  Batch size: 8
  Steps per epoch: 518
  Total steps: 518
  Warmup steps: 100

Training...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.000100



Model saved to: ./finetuned_narrative_model

[5/5] Evaluating fine-tuned model...

Evaluating Fine-tuned Model
Test set size: 200

✓ Test Accuracy: 0.6400 (64.00%)
  Correct: 128/200

FINAL COMPARISON
Baseline:    0.6150 (61.50%)
Fine-tuned:  0.6400 (64.00%)
Improvement: +2.50 percentage points

 Fine-tuning improved performance!


In [21]:
    import shutil
    import os

    folder_to_download = 'finetuned_narrative_model'
    zip_filename = f'{folder_to_download}.zip'

    # Create the zip archive
    shutil.make_archive(folder_to_download, 'zip', folder_to_download)

'/content/finetuned_narrative_model.zip'

In [20]:
# train_loss = losses.OnlineContrastiveLoss(model=model)
# # or
# train_loss = losses.MultipleNegativesRankingLoss(model=model)

In [28]:
def grid_search_weights_from_examples(baseline_model, finetuned_model, val_examples):
    best_acc = 0
    best_w = None

    for w_baseline in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        w_finetuned = 1.0 - w_baseline

        correct = 0
        for ex in val_examples:
            anchor = ex.texts[0]
            positive = ex.texts[1]  # Should be more similar
            negative = ex.texts[2]  # Should be less similar

            # Baseline
            baseline_embs = baseline_model.encode([anchor, positive, negative])
            sim_pos_baseline = cosine_similarity([baseline_embs[0]], [baseline_embs[1]])[0][0]
            sim_neg_baseline = cosine_similarity([baseline_embs[0]], [baseline_embs[2]])[0][0]

            # Fine-tuned
            finetuned_embs = finetuned_model.encode([anchor, positive, negative])
            sim_pos_finetuned = cosine_similarity([finetuned_embs[0]], [finetuned_embs[1]])[0][0]
            sim_neg_finetuned = cosine_similarity([finetuned_embs[0]], [finetuned_embs[2]])[0][0]

            # Combined
            sim_pos = w_baseline * sim_pos_baseline + w_finetuned * sim_pos_finetuned
            sim_neg = w_baseline * sim_neg_baseline + w_finetuned * sim_neg_finetuned

            # Predict: positive should have higher similarity
            if sim_pos > sim_neg:
                correct += 1

        acc = correct / len(val_examples)
        if acc > best_acc:
            best_acc = acc
            best_w = (w_baseline, w_finetuned)

        print(f"w_baseline={w_baseline:.1f}, w_finetuned={w_finetuned:.1f} → {acc:.2%}")

    return best_w

# Use directly with InputExamples
best_weights = grid_search_weights_from_examples(baseline_model, finetuned_model, val_examples)
print(f"\nBest: baseline={best_weights[0]:.1f}, finetuned={best_weights[1]:.1f}")

w_baseline=0.0, w_finetuned=1.0 → 65.53%
w_baseline=0.1, w_finetuned=0.9 → 64.30%
w_baseline=0.2, w_finetuned=0.8 → 63.89%
w_baseline=0.3, w_finetuned=0.7 → 63.34%
w_baseline=0.4, w_finetuned=0.6 → 63.47%
w_baseline=0.5, w_finetuned=0.5 → 62.79%
w_baseline=0.6, w_finetuned=0.4 → 62.24%
w_baseline=0.7, w_finetuned=0.3 → 62.24%
w_baseline=0.8, w_finetuned=0.2 → 62.11%
w_baseline=0.9, w_finetuned=0.1 → 61.70%
w_baseline=1.0, w_finetuned=0.0 → 61.42%

Best: baseline=0.0, finetuned=1.0


In [29]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def ensemble_predict(anchor, text_a, text_b, baseline_model, finetuned_model, weights=[0.5, 0.5]):
    """
    Combine predictions from baseline and fine-tuned models
    """
    # Get embeddings from baseline
    baseline_embs = baseline_model.encode([anchor, text_a, text_b])
    sim_a_baseline = cosine_similarity([baseline_embs[0]], [baseline_embs[1]])[0][0]
    sim_b_baseline = cosine_similarity([baseline_embs[0]], [baseline_embs[2]])[0][0]

    # Get embeddings from fine-tuned
    finetuned_embs = finetuned_model.encode([anchor, text_a, text_b])
    sim_a_finetuned = cosine_similarity([finetuned_embs[0]], [finetuned_embs[1]])[0][0]
    sim_b_finetuned = cosine_similarity([finetuned_embs[0]], [finetuned_embs[2]])[0][0]

    # Weighted combination
    sim_a_combined = weights[0] * sim_a_baseline + weights[1] * sim_a_finetuned
    sim_b_combined = weights[0] * sim_b_baseline + weights[1] * sim_b_finetuned

    return sim_a_combined > sim_b_combined

# Evaluate ensemble
baseline_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
finetuned_model = SentenceTransformer('./finetuned_narrative_model')

test_df1 = pd.read_json(test_data_path, lines=True)
test_df2 = pd.read_json('./sample_track_a.jsonl', lines=True)
test_df = pd.concat([test_df1, test_df2])
print(f"Test set size: {len(test_df)}")

correct = 0
for idx, row in test_df.iterrows():
    pred = ensemble_predict(
        row['anchor_text'],
        row['text_a'],
        row['text_b'],
        baseline_model,
        finetuned_model,
        weights=[0, 1]
    )
    if pred == row['text_a_is_closer']:
        correct += 1

accuracy = correct / len(test_df)
print(f"Ensemble accuracy: {accuracy:.2%}")

Test set size: 239
Ensemble accuracy: 63.18%
